# 02 — Experiments: 84-Run Simulation
**Project:** Optimizer and Learning Rate Study of Ovarian Cancer Prediction  
**Course:** HI 192 — Knowledge Representation and Health Decision Support

This is the master experiment notebook. It trains all **84 model configurations** (4 architectures × 7 optimizers × 3 learning rates) and saves every artifact to `results/`.

---
## Section 0 — Imports and Configuration

In [ ]:
import os
import sys
import pathlib
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Add project root to path so src/ modules are importable
PROJECT_ROOT = pathlib.Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model_builder import build_model
from src.train import train_model
from src.evaluate import evaluate_model
from src.visualize import plot_accuracy_loss, plot_auc_roc

# ── Global random seed ────────────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
PROCESSED_DIR = PROJECT_ROOT / 'dataset' / 'processed'
RESULTS_DIR   = PROJECT_ROOT / 'results'

METRICS_DIR      = RESULTS_DIR / 'metrics'
CURVES_ACC_DIR   = RESULTS_DIR / 'curves' / 'accuracy_loss'
CURVES_ROC_DIR   = RESULTS_DIR / 'curves' / 'auc_roc'
CM_DIR           = RESULTS_DIR / 'confusion_matrices'
SUMMARY_DIR      = RESULTS_DIR / 'summary'

for d in [METRICS_DIR, CURVES_ACC_DIR, CURVES_ROC_DIR, CM_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Experiment matrix ─────────────────────────────────────────────────────────
ARCHITECTURES  = ['VGG19', 'EfficientNetB3', 'ResNet50', 'DenseNet121']
OPTIMIZERS     = ['Adam', 'Adagrad', 'Adamax', 'AdaDelta', 'SGD', 'RMSProp', 'Nadam']
LEARNING_RATES = [1e-4, 1e-5, 1e-6]

# ── Fixed training hyperparameters ────────────────────────────────────────────
EPOCHS     = 50
BATCH_SIZE = 32
IMG_SIZE   = (256, 256)

total_runs = len(ARCHITECTURES) * len(OPTIMIZERS) * len(LEARNING_RATES)
print(f"Experiment matrix: {len(ARCHITECTURES)} arch × {len(OPTIMIZERS)} opt × {len(LEARNING_RATES)} LR = {total_runs} runs")

---
## Section 1 — Shared Data Pipeline

> ⚠️ **EXPERIMENTAL INTEGRITY NOTE:** The data split and generators defined in this section are constructed **once** and shared across all 84 simulation runs. Do **not** re-initialize generators inside the training loop. Re-initializing would reshuffle the training set differently per run, introducing data variability as a confound. All observed performance differences must be attributable solely to the choice of optimizer and learning rate.

In [ ]:
# ── Generator definitions (constructed ONCE here, reused across all 84 runs) ──

train_datagen = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.3,
    zoom_range=0.30,
    horizontal_flip=True,
    fill_mode='nearest',
)

eval_datagen = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True,
)

train_generator = train_datagen.flow_from_directory(
    PROCESSED_DIR / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=RANDOM_SEED,
)

val_generator = eval_datagen.flow_from_directory(
    PROCESSED_DIR / 'val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

test_generator = eval_datagen.flow_from_directory(
    PROCESSED_DIR / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

CLASS_NAMES = list(train_generator.class_indices.keys())
print(f"Train : {train_generator.samples} images")
print(f"Val   : {val_generator.samples} images")
print(f"Test  : {test_generator.samples} images")
print(f"Classes: {train_generator.class_indices}")

---
## Section 2 — Training Loop

Iterates over all 84 configurations. Each run:
1. Builds a fresh model via `model_builder.build_model()`
2. Trains via `train.train_model()`
3. Evaluates via `evaluate.evaluate_model()`
4. Saves accuracy/loss and AUC-ROC curves via `visualize`
5. Writes all artifacts under `results/{run_name}/`

**Run naming convention:** `{Architecture}_{Optimizer}_LR{lr}`  
e.g., `ResNet50_Adam_LR1e-4`

In [ ]:
all_metrics = []
run_number  = 0

for arch in ARCHITECTURES:
    for opt in OPTIMIZERS:
        for lr in LEARNING_RATES:
            run_number += 1
            lr_str   = f"{lr:.0e}".replace('e-0', 'e-').replace('e+0', 'e+')
            run_name = f"{arch}_{opt}_LR{lr_str}"

            print("\n" + "=" * 72)
            print(f"  Run {run_number:02d}/{total_runs}  |  {arch}  |  {opt}  |  LR={lr}")
            print("=" * 72)

            # ── 1. Build model ────────────────────────────────────────────────
            model = build_model(
                architecture_name=arch,
                optimizer_name=opt,
                learning_rate=lr,
            )

            # ── 2. Train ──────────────────────────────────────────────────────
            # NOTE: train_generator is reused (not re-instantiated) intentionally.
            history = train_model(
                model=model,
                train_generator=train_generator,
                val_generator=val_generator,
                epochs=EPOCHS,
                run_name=run_name,
            )

            # ── 3. Accuracy/loss and AUC-ROC curves ───────────────────────────
            plot_accuracy_loss(
                history=history,
                run_name=run_name,
                output_dir=str(CURVES_ACC_DIR),
            )
            plot_auc_roc(
                model=model,
                test_generator=test_generator,
                run_name=run_name,
                output_dir=str(CURVES_ROC_DIR),
            )

            # ── 4. Evaluate ───────────────────────────────────────────────────
            metrics = evaluate_model(
                model=model,
                test_generator=test_generator,
                run_name=run_name,
                output_dir=str(RESULTS_DIR),
            )
            all_metrics.append(metrics)

            # Free GPU memory before next run
            del model
            tf.keras.backend.clear_session()

print("\n" + "=" * 72)
print(f"  All {total_runs} runs complete.")
print("=" * 72)

---
## Section 3 — Aggregate Results

Load all per-run CSVs from `results/metrics/`, compile into one master DataFrame, and save as `results/summary/all_runs_summary.csv`.

In [ ]:
import glob

csv_files = glob.glob(str(METRICS_DIR / '*.csv'))
frames = [pd.read_csv(f) for f in sorted(csv_files)]
master_df = pd.concat(frames, ignore_index=True)

# Sort by AUC descending for a quick leaderboard view
master_df = master_df.sort_values('auc', ascending=False).reset_index(drop=True)

summary_path = SUMMARY_DIR / 'all_runs_summary.csv'
master_df.to_csv(summary_path, index=False)
print(f"Master summary saved → {summary_path}")
print(f"Total rows: {len(master_df)}")

display(master_df.head(20))